# Preprocessing

Keep the available movie records and make the fields used by the recommender easier to work with.

In [1]:
from pathlib import Path
import json
import pandas as pd

data_path = "../data/original_dataset.csv"
output_path = "../data/cleaned_data.csv"
# Load the original movie data
movies = pd.read_csv(data_path)
print("Before cleaning:", movies.shape)
print("Exact duplicate rows:", int(movies.duplicated().sum()))
print("Missing titles:", int(movies["title"].isna().sum()))

Before cleaning: (4803, 20)
Exact duplicate rows: 0
Missing titles: 0


## Clean text and parse metadata

In [2]:
def names_from_json(value, separator=" "):
    if pd.isna(value) or not str(value).strip():
        return ""
    try:
        records = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return ""
    if not isinstance(records, list):
        return ""
    names = [str(row["name"]).strip() for row in records
             if isinstance(row, dict) and row.get("name") and str(row["name"]).strip()]
    return separator.join(names)

# Remove duplicate rows and records without a usable title
movies = movies.drop_duplicates().copy()
movies = movies[movies["title"].notna()].copy()
movies["title"] = movies["title"].astype(str).str.strip()
movies = movies[movies["title"] != ""].copy()

# Keep missing text fields usable in the recommender
for column in ["overview", "tagline"]:
    movies[column] = movies[column].fillna("").astype(str).str.strip()

# Turn the JSON metadata into readable names
movies["genres_text"] = movies["genres"].map(lambda value: names_from_json(value, " | "))
movies["keywords_text"] = movies["keywords"].map(names_from_json)
# Convert date and numeric fields
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["release_year"] = movies["release_date"].dt.year

for column in ["budget", "revenue", "popularity", "runtime", "vote_average", "vote_count"]:
    movies[column] = pd.to_numeric(movies[column], errors="coerce")

# Check the cleaned data
print("After cleaning:", movies.shape)
print("Missing text values in overview/tagline:", int(movies[["overview", "tagline"]].isna().sum().sum()))
display(movies[["title", "genres_text", "keywords_text", "release_year"]].head())

After cleaning: (4803, 23)
Missing text values in overview/tagline: 0


,title,genres_text,keywords_text,release_year
0,Avatar,Action | Adventure | Fantasy | Science Fiction,culture clash future space war space colony so...,2009.0
1,Pirates of the Caribbean: At World's End,Adventure | Fantasy | Action,ocean drug abuse exotic island east india trad...,2007.0
2,Spectre,Action | Adventure | Crime,spy based on novel secret agent sequel mi6 bri...,2015.0
3,The Dark Knight Rises,Action | Crime | Drama | Thriller,dc comics crime fighter terrorist secret ident...,2012.0
4,John Carter,Action | Adventure | Science Fiction,based on novel mars medallion space travel pri...,2012.0


## Save cleaned data

In [3]:
# Save the cleaned dataset
movies.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Remaining missing values in selected text:", int(movies[["title", "genres_text", "keywords_text", "overview", "tagline"]].isna().sum().sum()))

Saved: ../data/cleaned_data.csv
Remaining missing values in selected text: 0


## Before and after

- Rows: 4,803 before and 4,803 after removing exact duplicate rows and blank titles.
- Exact duplicate rows found: 0; missing or blank titles found: 0.
- Missing overview and tagline values are represented as empty text. Genres and keywords are parsed into readable name fields.
- Numeric fields are converted to numeric types, and `release_year` is derived from `release_date`. Other source columns are retained.